# 04 — Modele ARMA pentru randamente

Pentru fiecare activ (`Rand_SNP`, `Rand_SNN`, `Rand_H2O`, `Rand_BET-NG`):

1. split cronologic 80% train / 20% test;
2. estimare ARMA(p, q), p, q ≤ 2, pe setul de antrenament;
3. selecție: cel mai mic AIC dintre modelele cu **toți coeficienții AR/MA semnificativi** (p < 0.05); dacă niciunul nu îndeplinește condiția, se alege zgomotul alb ARMA(0,0);
4. diagnostic pe reziduuri: Ljung-Box (autocorelare) și ARCH-LM (efecte ARCH → justifică modelele GARCH).

In [1]:
import sys
import warnings
from pathlib import Path

# Face pachetul `src` importabil indiferent din ce folder se pornește notebookul
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.arima.model import ARIMA

from src import config, data
from src.garchx import split_train

df = data.load_master_dataset()

CANDIDATI = [(0, 0, 0), (1, 0, 0), (0, 0, 1), (1, 0, 1), (2, 0, 0), (0, 0, 2), (2, 0, 1), (1, 0, 2), (2, 0, 2)]
LAGS_LJUNG = [1, 5, 10, 20, 30]
LAGS_ARCH = [1, 4, 8, 12]

In [2]:
modele_alese, ljung_box, arch_lm = [], [], []
ordine_aleasa = {}

for var in config.ASSETS:
    train, test = split_train(df[[var]].dropna())
    train = train[var]
    print(f"{var}: {len(train) + len(test)} observații | train {len(train)} | test {len(test)}")

    # --- ACF / PACF pe setul de antrenament
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(train, lags=36, ax=axes[0], title=f"ACF - Train Set ({var})")
    plot_pacf(train, lags=36, ax=axes[1], title=f"PACF - Train Set ({var})", method="ywm")
    plt.tight_layout()
    plt.savefig(config.FIG_DIR / f"arma_acf_pacf_train_{var}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # --- Estimarea modelelor candidate
    candidati, ordine = [], {}
    for ordin in CANDIDATI:
        try:
            fit = ARIMA(train, order=ordin).fit()
        except Exception:
            continue
        pvals = fit.pvalues.drop(["const", "sigma2"], errors="ignore")
        eticheta = f"ARMA({ordin[0]},{ordin[2]})"
        ordine[eticheta] = ordin
        candidati.append({"Model": eticheta, "AIC": fit.aic, "BIC": fit.bic,
                          "Toti Coef. Semnificativi?": "DA" if (pvals < 0.05).all() else "NU"})
    df_candidati = pd.DataFrame(candidati)
    df_candidati.to_csv(config.TABLE_DIR / f"arma_candidates_{var}.csv", index=False)

    # --- Selecția modelului
    valide = df_candidati[df_candidati["Toti Coef. Semnificativi?"] == "DA"].sort_values("AIC")
    if len(valide):
        eticheta_aleasa, aic = valide.iloc[0]["Model"], valide.iloc[0]["AIC"]
    else:
        eticheta_aleasa = "ARMA(0,0)"
        aic = df_candidati.loc[df_candidati["Model"] == eticheta_aleasa, "AIC"].iloc[0]
    ordine_aleasa[var] = ordine[eticheta_aleasa]
    print(f"  model selectat: {eticheta_aleasa}")

    fit = ARIMA(train, order=ordine[eticheta_aleasa]).fit()
    rezid = fit.resid
    modele_alese.append({"Activ": var, "Model Selectat": eticheta_aleasa, "AIC": round(aic, 2),
                         "Dimensiune Train": len(train), "Dimensiune Test": len(test)})

    # --- Diagnostic: Ljung-Box și ARCH-LM pe reziduuri
    lb = acorr_ljungbox(rezid, lags=LAGS_LJUNG, return_df=True)
    for lag in LAGS_LJUNG:
        ljung_box.append({"Activ": var, "Model": eticheta_aleasa, "Lag": lag,
                          "Statistica Q": round(lb.loc[lag, "lb_stat"], 4),
                          "p-value": round(lb.loc[lag, "lb_pvalue"], 4),
                          "Autocorelare?": "NU (E Zgomot Alb)" if lb.loc[lag, "lb_pvalue"] > 0.05 else "DA"})
    for lag in LAGS_ARCH:
        stat, p, _, _ = het_arch(rezid, nlags=lag)
        arch_lm.append({"Activ": var, "Model": eticheta_aleasa, "Lag ARCH": lag,
                        "Statistica LM": round(stat, 4),
                        "p-value": "< 0.0001" if p < 0.0001 else f"{p:.4f}",
                        "Efecte ARCH? (Necesită GARCH)": "DA" if p < 0.05 else "NU"})

    # --- PACF reziduuri la pătrat (ordinul GARCH)
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_pacf(rezid ** 2, lags=12, ax=ax, title=f"PACF - Reziduuri la pătrat ({var} - {eticheta_aleasa})", method="ywm")
    plt.tight_layout()
    plt.savefig(config.FIG_DIR / f"arma_pacf_sq_resid_{var}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

df_modele = pd.DataFrame(modele_alese)
df_modele.to_csv(config.TABLE_DIR / "arma_selected_models.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(ljung_box).to_csv(config.TABLE_DIR / "arma_ljung_box.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(arch_lm).to_csv(config.TABLE_DIR / "arma_arch_lm.csv", index=False, encoding="utf-8-sig")
df_modele

Rand_SNP: 3844 observații | train 3075 | test 769


c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\User

  model selectat: ARMA(2,1)


c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)


Rand_SNN: 3381 observații | train 2704 | test 677


...

  model selectat: ARMA(0,0)
Rand_BET-NG: 3844 observații | train 3075 | test 769


c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\User

  model selectat: ARMA(2,0)


c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\paulc\Downloads\bvb-energy-volatility-forecasting\bvb-energy-volatility-forecasting\.venv312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)


,Activ,Model Selectat,AIC,Dimensiune Train,Dimensiune Test
0,Rand_SNP,"ARMA(2,1)",-17255.85,3075,769
1,Rand_SNN,"ARMA(2,0)",-15043.85,2704,677
2,Rand_H2O,"ARMA(0,0)",-4437.22,668,167
3,Rand_BET-NG,"ARMA(2,0)",-19673.67,3075,769


## Verificare: ordinele selectate vs. cele folosite în etapele următoare

Notebookurile 05–07 folosesc ordinele ARMA din `src/config.py`. Celula de mai jos arată dacă selecția de aici coincide cu acestea.

In [3]:
for var in config.ASSETS:
    gasit, folosit = ordine_aleasa[var], config.ARMA_ORDERS[var]
    stare = "OK" if gasit == folosit else "DIFERIT"
    print(f"{var:12s} selectat aici: {gasit} | folosit în config: {folosit} -> {stare}")

Rand_SNP     selectat aici: (2, 0, 1) | folosit în config: (2, 0, 1) -> OK
Rand_SNN     selectat aici: (2, 0, 0) | folosit în config: (2, 0, 0) -> OK
Rand_H2O     selectat aici: (0, 0, 0) | folosit în config: (0, 0, 0) -> OK
Rand_BET-NG  selectat aici: (2, 0, 0) | folosit în config: (2, 0, 0) -> OK
